In [21]:
from dragonfly import Window
from dragonfly.windows.rectangle   import Rectangle
from dragonfly.windows.darwin_window import DarwinWindow
from dragonfly.windows.win32_window import Win32Window
from dataclasses import dataclass
import time

In [2]:
@dataclass
class WindowInfo:
    title: str
    executable: str
    handle: int
    window: Win32Window | DarwinWindow # assume we only handle macOS and Windows

In [3]:
def is_window_important(window: Win32Window | DarwinWindow) -> bool:
    # ignore windows without a title or executable because these are usually system processes
    if not window.title or not window.executable:
        return False
    
    if isinstance(window, Win32Window):
        # ignore windows that are not enabled (cannot receive mouse/keyboard input)
        if not window.is_enabled:
            return False
        
        # ignore windows that are not visible because they are usually not user facing apps
        if not window.is_visible:
            return False

        # ignore windows that are part of the OS
        # user applications are usually in C:\Program Files\ or /Applications
        if window.executable.startswith("C:\\Windows\\"):
            return False
        
    
    elif isinstance(window, DarwinWindow):
        # TODO: write macOS filtering logic as needed
        pass # do nothing for now
    
    return True

In [4]:
def get_window_list() -> list[WindowInfo]:
    windows = Window.get_all_windows()
    window_info: list[WindowInfo] = []

    for window in windows:
        if not is_window_important(window):
            continue
        try:
            info = {
                'title': window.title,
                'executable': window.executable,
                'handle': window.handle,
                'window': window
            }
            window_info.append(info)
        except Exception as e:
            print(f"Error getting info for window: {e}")

    return window_info

In [ ]:
window_list = get_window_list()
for window in window_list:
    print(f"Window: {window['title']}")
    print(f"  Executable: {window['executable']}")
    print(f"  Handle: {window['handle']}")
    print(f"  is_visible: {window['window'].is_visible} | is_minimized: {window['window'].is_minimized} | is_maximized: {window['window'].is_maximized} | is_valid: {window['window'].is_valid}")
    print("-" * 50)

    

In [63]:
window2: Win32Window = window_list[0]['window']

for window in window_list:
    if window['title'].endswith("Discord.exe"):
        window2 = window['window']
        break

In [10]:
window_shake = window_list[3]['window']
print(window_shake.title)

GitHub Desktop


In [56]:
import math, random

In [161]:
# using the starting position, execute a series of window movements that make the window appear like it is shaking for a second
starting_position = window_shake.get_position()
start_l, start_t, start_w, start_h = starting_position.ltwh

if window_shake.is_maximized:
    window_shake.restore()
    window_shake.set_position(Rectangle(0, 0, start_w, start_h))
    time.sleep(0.3)


shake_iterations = 20
intensity = 20
duration = 0.4
interval = duration / shake_iterations

end_time = time.time() + duration

# Repeatedly update the window's position.
while time.time() < end_time:
    # Calculate a random offset (you can add damping if you want a decaying shake).
    dx = random.randint(-intensity, intensity)
    dy = random.randint(-intensity, intensity)

    # Update the window's position.
    window_shake.set_position(Rectangle(start_l + dx, start_t + dy,
                        start_w + dx, start_h + dy))

    time.sleep(interval)

window_shake.set_position(starting_position)
window_shake.maximize()

24

In [152]:
window_shake.get_position()

Rectangle(-10.0, 0.0, 1926.0, 1128.0)

In [155]:
window_shake.get_position()

Rectangle(229.0, 75.0, 1440.0, 990.0)